# CP-SAT 简介：用 SAT 的技术做约束编程

## 这一章讲什么

OR-Tools 的 **CP-SAT** 是 Google 做的约束编程求解器。它和 Z3 那几章一样，都是“把条件写下来、让程序去找解”的工具，只是它更擅长**最优化**，比如让利润最大、让距离最短。这一章我们看四件事。

- CP-SAT 底下靠什么在解（SAT 的技术，加上线性松弛）
- 最基本的用法：`CpModel` / `new_int_var` / `add` / `solve`
- 一道线性规划的例题，把利润做到最大
- 用 `assumptions` 做一道推理题

## CP-SAT 是什么

CP-SAT 是**拿 SAT（可满足性问题）的技术当底子的约束编程求解器**。

- 搜索怎么往前走？靠的是 SAT 求解器那套 **CDCL**（撞了墙就学着剪枝）和 **LCG**（Lazy Clause Generation，惰性子句生成）。
- 它还把**线性松弛**拼了进来，所以像 MIP（混合整数规划）求解器那样“把目标函数做到最大或最小”，它也做得了。

OR-Tools 里其实早就有 `constraint_solver`，那是经典 CP 求解器，以深度优先搜索为主，最优化不是它的强项。所以**现在用 OR-Tools，就选 CP-SAT**，这基本算定论。本书的 OrTools 章节，一律用 CP-SAT。

(ortools-send-more-money)=
## 例题：SEND + MORE = MONEY

[用 Z3 求解 SEND+MORE=MONEY](z3-send-more-money)

这道字母算术在 Z3 那章露过一次面，这回换 CP-SAT 再解一遍。用法的整个流程，都装在这一段代码里。

In [1]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()
s, e, n, d, m, o, r, y = [model.new_int_var(0, 10, name) for name in "sendmory"]
model.add(s != 0)
model.add(m != 0)
model.add_all_different([s, e, n, d, m, o, r, y])
send = s * 1000 + e * 100 + n * 10 + d
more = m * 1000 + o * 100 + r * 10 + e
money = m * 10000 + o * 1000 + n * 100 + e * 10 + y

model.add(send + more == money)
solver = cp_model.CpSolver()
solver.Solve(model)
print(f"{solver.value(send)} + {solver.value(more)} = {solver.value(money)}")

9567 + 1085 = 10652


### 这段代码在干什么

- `CpModel()` 建一个模型，也就是装约束的那个容器
- `model.new_int_var(下限, 上限, 名字)` 声明一个整数变量
- `model.add(条件)` 往里添约束，`!=`, `==`, `>=` 这些比较式原样写就行
- `model.add_all_different([...])` 要求这些变量取值互不相同
- `solver.solve(model)` 交给求解器去算
- `solver.value(式)` 把式子的值取出来

式子可以直接用 `*` 和 `+` 拼，所以像 `s * 1000 + e * 100 + ...` 这样，把数位照抄下来就成。要写加权和，`LinearExpr.weighted_sum()` 也可以。

```python
send = cp_model.LinearExpr.weighted_sum([s, e, n, d], [1000, 100, 10, 1])
more = cp_model.LinearExpr.weighted_sum([m, o, r, e], [1000, 100, 10, 1])
money = cp_model.LinearExpr.weighted_sum([m, o, n, e, y], [10000, 1000, 100, 10, 1])
```

## 线性规划

接下来换个口味，做一道“把利润做到最大”的题。先看线性规划的标准例子。

要生产产品 A 和 B，需要三种原料 a, b, c。每做 1 单位的用量和库存列在下面；卖出 1 单位，A 赚 3 万日元，B 赚 2 万日元。各生产多少，利润最大？

| 原料 | 产品 A | 产品 B | 库存 |
|------|--------|--------|------|
| a    | 3      | 1      | 9    |
| b    | 2.5    | 2      | 12.5 |
| c    | 1      | 2      | 8    |

设产量为 $x_1$, $x_2$，这道题就能写成下面这样。

**目标函数**

$$
z = 3x_1 + 2x_2 \quad \text{（最大化）}
$$

**约束条件**

$$
\begin{aligned}
3x_1 + x_2 &\leq 9 \\
2.5x_1 + 2x_2 &\leq 12.5 \\
x_1 + 2x_2 &\leq 8 \\
x_1, x_2 &\geq 0
\end{aligned}
$$

CP-SAT 只认**整数**。式子里冒出一个 `2.5` 怎么办？两边一起乘 2，把小数赶出去（`2.5x1 + 2x2 <= 12.5` → `5x1 + 4x2 <= 25`）。不等式两边同乘一个正数，条件的意思不变。

`model.maximize(z)` 说的就是“去找让 $z = 3x_1 + 2x_2$ 取最大值的 $x_1, x_2$”。

In [2]:
model = cp_model.CpModel()
x1, x2 = [model.new_int_var(0, 10000, f"x{i}") for i in range(1, 3)]
model.add(3 * x1 + x2 <= 9)
model.add(5 * x1 + 4 * x2 <= 25)
model.add(x1 + 2 * x2 <= 8)
z = 3 * x1 + 2 * x2
model.maximize(z)
solver = cp_model.CpSolver()
solver.Solve(model)
print(f"x1 = {solver.value(x1)}, x2 = {solver.value(x2)}, z = {solver.value(z)}")

x1 = 2, x2 = 3, z = 12


## 逻辑谜题

有红、蓝、黄、绿四个箱子，其中几个里头放着苹果。至少有一个，两个、三个放也都没关系。

你可以挑一个箱子，把里面的东西拿走。问题是：哪一只一定装着苹果？线索只有两条。

- 红箱子里有苹果的话，蓝箱子里也有。
- 绿箱子里没有苹果的话，黄箱子和蓝箱子里都没有。

在 CP-SAT 里，布尔变量加逻辑约束，这两句话能照原样写下去，剩下的推理由它来做。

In [3]:
model = cp_model.CpModel()
red, blue, yellow, green = [model.new_bool_var(name) for name in ['red', 'blue', 'yellow', 'green']]

model.add_bool_or([red, blue, yellow, green])                 # 至少有 1 个
model.add_implication(red, blue)                               # 红 → 蓝
model.add_bool_and(~yellow, ~blue).only_enforce_if(~green)     # 没有绿 → 黄和蓝也没有

solver = cp_model.CpSolver()
for box in [red, blue, yellow, green]:
    model.add_assumption(~box)          # 假设“这个箱子里没有”
    res = solver.solve(model)
    if res == cp_model.INFEASIBLE:      # 矛盾 → 一定放着
        print(box)
    model.clear_assumptions()           # 删掉假设

green


### 这几行代码在做什么

- `new_bool_var()` 造一个 0/1 的布尔变量
- `add_bool_or([...])` 至少 1 个为真（逻辑或）
- `add_implication(a, b)` 若 `a` 则 `b`（逻辑蕴含）
- `add_bool_and(a, b).only_enforce_if(c)` 只在 `c` 成立时要求“`a` 且 `b`”
  - `~` 就是取反
  - `only_enforce_if()` 指定“只在这个条件成立时才有效”

这道谜题要的不是随便找一个解，而是推理出**哪个箱子一定装着苹果**。所以对每个箱子，我们先假设“里面没有”，再去解；**如果矛盾（`INFEASIBLE`），那这个箱子里就一定有苹果**。

- 用 `model.add_assumption(条件)` 加假设
- 用 `clear_assumptions()` 把假设清掉（试下一个箱子之前，一定要调）

结果落在 `green` 上。想想为什么：没有绿，黄和蓝也就都没有，红→蓝这条又用不上，于是一个苹果都剩不下。

## 小结

- CP-SAT 干活靠两样：SAT 的技术，加线性松弛。用 OR-Tools 就选它
- 基本流程：`CpModel()` → 变量 → `add()` → `solve()` → `value()`
- 要最优化，用 `model.maximize(式)` / `model.minimize(式)`
- CP-SAT 只处理整数；式子里有小数，两边同乘一个整数把它消掉
- 布尔变量配逻辑约束（`add_bool_or`, `add_implication`, `only_enforce_if`），条件分支也写得出来
- `assumptions` 用来问“再加上这个条件，会不会矛盾”，推理谜题就靠它

下一章，我们把 CP-SAT 的 API 摊开细看。